# 01 — Exploratory Data Analysis

## Project Overview

This notebook explores the AG News dataset, which contains news articles from four categories: World, Sports, Business, and Sci/Tech.

The goal is to understand the dataset's structure, class distribution, text quality, and text length before building machine learning models.

## Objectives

- Inspect the dataset structure and available features.
- Examine the distribution of news categories.
- Review representative text samples from each category.
- Check for empty texts and duplicate articles.
- Analyze text length statistics.
- Compare average text lengths across categories.

## Dataset

The AG News dataset is a text classification dataset. Each example contains a news article and its corresponding category label.

The dataset will be used to compare a traditional machine learning baseline based on TF-IDF and Logistic Regression with a fine-tuned DistilBERT model.

In [1]:
import sys

print(sys.executable)

/mnt/d809bc61-ebac-4ca7-8e1e-e0a6d7e8f201/Documents/Programming/MyProjects/transformer-news-classification/.venv/bin/python


## 1. Loading the Dataset

The first step is to load the AG News dataset using the project's data-loading utility.

The dataset contains two splits: a training set with 120,000 articles and a test set with 7,600 articles. The training set is used to develop machine learning models, while the test set is reserved for final evaluation.

Inspecting the dataset object helps us understand its structure before analyzing individual samples.

**Key questions:**
- What splits are available?
- What information does each example contain?
- How are the news articles and labels represented?

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_ag_news

dataset = load_ag_news()

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

## 2. Class Distribution

AG News contains four news categories:

- **World**
- **Sports**
- **Business**
- **Sci/Tech**

We examine the number of samples belonging to each category in both the training and test sets.

Class distribution matters because a strongly imbalanced dataset can influence model training and make overall accuracy less informative.

Comparing the training and test distributions also helps us check whether the two splits have similar class proportions.

In [3]:
from collections import Counter

train_labels = dataset["train"]["label"]
test_labels = dataset["test"]["label"]

print("Train:", Counter(train_labels))
print("Test:", Counter(test_labels))

Train: Counter({2: 30000, 3: 30000, 1: 30000, 0: 30000})
Test: Counter({2: 1900, 3: 1900, 1: 1900, 0: 1900})


## 3. Dataset Features and Label Encoding

Before inspecting individual articles, we examine the dataset's available features and their data types.

The `text` feature contains the news article, while the `label` feature represents its category.

Understanding how labels are encoded is important because machine learning models operate on numeric labels, whereas humans interpret the corresponding category names.

We will use the category names provided by the dataset rather than manually assuming a label mapping.

In [4]:
print(dataset["train"].features)

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


## 4. Inspecting Representative News Articles

We inspect three training examples from each news category.

Reading actual articles helps us understand the type of information associated with each label and identify potential differences between categories.

This qualitative inspection is useful because two categories may discuss related subjects. For example, Business and Sci/Tech articles may sometimes contain overlapping terminology.

The examples also help us verify that the text and labels appear consistent.

In [5]:
label_names = dataset["train"].features["label"].names

for label in range(len(label_names)):
    print("=" * 80)
    print(f"CLASS {label}: {label_names[label]}")
    print("=" * 80)

    shown = 0

    for example in dataset["train"]:
        if example["label"] == label:
            shown += 1
            print(f"\nExample {shown}:")
            print(example["text"])

            if shown == 3:
                break

CLASS 0: World

Example 1:
Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\and in large numbers on Sunday to vote in a historic referendum\that will either remove left-wing President Hugo Chavez from\office or give him a new mandate to govern for the next two\years.

Example 2:
S.Koreans Clash with Police on Iraq Troop Dispatch (Reuters) Reuters - South Korean police used water cannon in\central Seoul Sunday to disperse at least 7,000 protesters\urging the government to reverse a controversial decision to\send more troops to Iraq.

Example 3:
Palestinians in Israeli Jails Start Hunger Strike (Reuters) Reuters - Thousands of Palestinian\prisoners in Israeli jails began a hunger strike for better\conditions Sunday, but Israel's security minister said he\didn't care if they starved to death.
CLASS 1: Sports

Example 1:
Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter free

## 5. Checking for Empty Texts

Text classification models require meaningful textual input.

Empty or missing articles may cause problems during preprocessing, feature extraction, or model inference.

We therefore count the training examples whose text is either missing (`None`) or contains only whitespace.

This check helps us determine whether the dataset requires additional handling before model training.

In [6]:
train_texts = dataset["train"]["text"]

empty_count = sum(
    1 for text in train_texts
    if text is None or not text.strip()
)

print("Total training samples:", len(train_texts))
print("Empty or None texts:", empty_count)

Total training samples: 120000
Empty or None texts: 0


## 6. Checking for Duplicate Articles

Duplicate articles may affect model evaluation if identical or highly similar examples appear in different dataset splits.

We count the number of unique training texts and estimate the number of exact duplicate entries.

This check identifies exact text repetition. It does not detect near-duplicates, such as articles that differ only slightly in wording.

The results provide an initial understanding of data redundancy before model development.

In [7]:
unique_texts = len(set(train_texts))
duplicate_count = len(train_texts) - unique_texts

print("Total training samples:", len(train_texts))
print("Unique texts:", unique_texts)
print("Duplicate texts:", duplicate_count)

Total training samples: 120000
Unique texts: 120000
Duplicate texts: 0


## 7. Text Length Analysis

News articles can vary considerably in length.

We calculate the minimum, maximum, and average number of characters across the training articles to understand the general range of text lengths.

Text length is particularly relevant when preparing data for Transformer models, because tokenization and maximum sequence length determine how much of each article the model can process.

These statistics provide an initial overview of the dataset's text characteristics.

In [8]:
import numpy as np

text_lengths = np.array([len(text) for text in train_texts])

print("Minimum characters:", np.min(text_lengths))
print("Maximum characters:", np.max(text_lengths))
print("Average characters:", np.mean(text_lengths))

Minimum characters: 100
Maximum characters: 1012
Average characters: 236.477525


## 8. Text Length Percentiles

The mean and maximum length do not fully describe the distribution of article lengths.

We therefore calculate the 50th, 90th, 95th, and 99th percentiles.

These percentiles help us understand how long a typical article is and how much longer the articles in the upper end of the distribution can be.

This information can guide later decisions about tokenization and sequence truncation for Transformer models.

**Note:** These measurements are based on characters, not tokens. Token lengths will be examined separately during Transformer preprocessing.

In [9]:
import numpy as np

percentiles = np.percentile(
    text_lengths,
    [50, 90, 95, 99]
)

print("50th percentile:", percentiles[0])
print("90th percentile:", percentiles[1])
print("95th percentile:", percentiles[2])
print("99th percentile:", percentiles[3])

50th percentile: 232.0
90th percentile: 300.0
95th percentile: 343.0
99th percentile: 464.0


## 9. Average Text Length by Category

Different news categories may have different writing styles and typical article lengths.

We calculate the average character length for each category to investigate whether these differences exist in the training data.

This analysis may reveal useful characteristics of the dataset, but average length alone does not indicate how difficult a category will be to classify.

The results should be interpreted alongside the actual article examples and the model evaluation results.

In [10]:
labels = dataset["train"]["label"]

for label, name in enumerate(label_names):
    class_lengths = text_lengths[np.array(labels) == label]

    print(
        name,
        "-> Average characters:",
        np.mean(class_lengths)
    )

World -> Average characters: 242.62666666666667
Sports -> Average characters: 224.6522
Business -> Average characters: 241.40856666666667
Sci/Tech -> Average characters: 237.22266666666667


# Conclusion

This notebook provided an initial exploration of the AG News dataset before model development.

The analysis covered:

- Dataset structure and available features.
- Training and test class distributions.
- Representative news articles from all four categories.
- Empty and duplicate training texts.
- Character-level text length statistics and percentiles.
- Average text length across news categories.

These checks help establish a better understanding of the data before building classification models.

The next notebook develops a TF-IDF and Logistic Regression baseline. This baseline will provide a reference point for evaluating the performance of a fine-tuned Transformer model later in the project.